#  Learning Rate Schedules & Adaptive Optimizers

## 1. The Learning Rate Problem
The **Learning Rate** ($\alpha$) is arguably the single most important hyperparameter in deep learning. It determines the size of the steps the optimizer takes downhill during Gradient Descent.
*  **Too High:** The model overshoots the minimum and diverges (explodes).
*  **Too Low:** The model takes microscopic steps, training takes weeks, and it easily gets stuck in shallow local minima.

**The Solution:** We don't want a static learning rate. We want it to start large (to learn fast and escape local minima) and get smaller over time (to carefully settle into the absolute lowest point). 

---

## 2. Learning Rate Schedules (Decay)
A schedule manually reduces the global learning rate over time based on the epoch number.



### A. Step Decay
Drops the learning rate by a specific factor after a set number of epochs.
* **Example:** Halve the learning rate every 10 epochs.
* **The Math:** $\alpha_t = \alpha_0 \cdot d^{\lfloor \frac{t}{s} \rfloor}$ 
*(Where $d$ is the drop rate, $t$ is the current epoch, and $s$ is the step size).*

### B. Exponential Decay
Smoothly reduces the learning rate using an exponential curve.
* **The Math:** $\alpha_t = \alpha_0 \cdot e^{-k t}$
*(Where $k$ is a decay constant).*

### C. Cosine Annealing
Reduces the learning rate following the shape of a cosine curve, often paired with "warm restarts" (bumping it back up periodically to escape local minima).

---

## 3. Adaptive Optimizers (The Modern Standard)
**The Flaw with Schedules:** Standard decay applies the *exact same learning rate* to every single weight in the network. But what if one feature is incredibly rare, while another appears constantly? We want an optimizer that gives **every individual weight its own custom learning rate**.



### A. AdaGrad (Adaptive Gradient Algorithm)
AdaGrad keeps a running sum of the squared gradients for each weight. It aggressively shrinks the learning rate for weights that receive large updates, and keeps it high for weights that rarely update (great for sparse data like NLP).
* **The Fatal Flaw:** Because it constantly adds positive numbers to its running sum, the denominator grows infinitely. Eventually, the learning rate shrinks to exactly **0**, and the network permanently stops learning.

### B. RMSProp (Root Mean Square Propagation)
Invented by Geoffrey Hinton (and famously published in a Coursera slide rather than a paper), RMSProp fixes AdaGrad's fatal flaw. Instead of keeping a cumulative sum of all past gradients, it uses an **Exponential Moving Average**. This means it gradually "forgets" older gradients, preventing the learning rate from shrinking to zero.

**The Math (per weight):**
1. Update the moving average of the squared gradient ($v_t$):
   $$v_t = \beta \cdot v_{t-1} + (1 - \beta) \cdot g_t^2$$
   *(Where $g_t$ is the current gradient, and $\beta$ is a decay term, usually **0.9**).*
2. Update the weight:
   $$W_{t+1} = W_t - \frac{\alpha}{\sqrt{v_t + \epsilon}} \cdot g_t$$
   *(Where $\epsilon$ is a tiny number to prevent dividing by zero).*

### C. Adam (Adaptive Moment Estimation)
Adam is the undisputed king of deep learning. It essentially merges **Momentum** (which builds up speed in consistent directions) with **RMSProp** (which adapts the learning rate per weight).

**The Math (The Engine of Modern AI):**
1. **First Moment (Momentum):** Calculate the moving average of the gradient.
   $$m_t = \beta_1 \cdot m_{t-1} + (1 - \beta_1) \cdot g_t$$
2. **Second Moment (RMSProp):** Calculate the moving average of the squared gradient.
   $$v_t = \beta_2 \cdot v_{t-1} + (1 - \beta_2) \cdot g_t^2$$
3. **Bias Correction:** Because $m$ and $v$ are initialized at `0`, they are biased toward zero at the start of training. Adam mathematically corrects this warm-up phase:
   $$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$
4. **Final Weight Update:**
   $$W_{t+1} = W_t - \frac{\alpha}{\sqrt{\hat{v}_t} + \epsilon} \cdot \hat{m}_t$$

---

## 4. Summary Cheat Sheet

| Optimizer / Method | How it Works | When to Use |
| :--- | :--- | :--- |
| **SGD** | Standard Gradient Descent. | Baseline. Slow and easily stuck without Momentum. |
| **LR Decay / Schedules** | Manually drops the global LR over time. | Fine-tuning late in training; training ResNets. |
| **AdaGrad** | Adapts LR per weight based on total past gradients. | Sparse data, but generally obsolete due to LR death. |
| **RMSProp** | Adapts LR using a moving average of squared gradients. | Excellent for Recurrent Neural Networks (RNNs). |
| **Adam** | Combines Momentum + RMSProp + Bias Correction. | **The Default Choice.** Start almost every project here. |

---

##  Key Takeaways
**Why it matters:** A static learning rate is highly inefficient. We must adapt the speed to the terrain of the loss landscape.  

**Schedules:** Adjust the global speed limit based on time/epochs. 

**Adaptive Methods:** Give every single weight its own personalized speed limit based on its historical behavior.  

**The Industry Standard:** If you are unsure which optimizer to use, blindly pick **Adam** with a learning rate of **0.001**. It works exceptionally well out of the box for 95% of architectures.